In [ ]:
!pip install transformers accelerate torch pandas


In [ ]:
import random
import re
import pandas as pd
import torch
from transformers import pipeline

print("Loading model...")
generator = pipeline(
    "text-generation",
    model="HuggingFaceH4/zephyr-7b-beta",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("Model loaded successfully!")

Loading model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
import pandas as pd
import random
import torch
from transformers import pipeline
from tqdm import tqdm

print("--- 1. Initializing Local LLM for Patient Persona ---")
# We use a fast local model to generate the natural language portal messages
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16
)



--- 1. Initializing Local LLM for Patient Persona ---


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [ ]:
print("--- 2. Defining the Clinical PMOS Logic (From the Lancet Guidelines) ---")

# We define the fragmented symptoms across different specialties
symptom_pools = {
    "HIGH": {
        "Dermatology": ["Severe cystic acne on jawline", "Noticeable hair thinning", "Acanthosis nigricans (dark skin patches)"],
        "OBGYN": ["Missed periods for 4-6 months", "Irregular heavy bleeding", "Ovarian cysts observed on ultrasound"],
        "Endocrinology/Family": ["Rapid unexplained weight gain (10kg+)", "Severe daily fatigue", "Borderline high HbA1c (insulin resistance)"]
    },
    "MEDIUM": {
        "Dermatology": ["Mild breakouts on chin", "Oily skin"],
        "OBGYN": ["Periods occasionally late (35-40 day cycles)", "Painful cramps"],
        "Endocrinology/Family": ["Feeling tired after meals", "Difficulty losing weight", "Sugar cravings"]
    },
    "LOW": {
        "Dermatology": ["Occasional dry skin", "Sunburn", "Normal skin check"],
        "OBGYN": ["Regular 28-day cycles", "Routine pap smear normal", "Refill birth control"],
        "Endocrinology/Family": ["Sprained ankle", "Seasonal allergies", "Mild cold symptoms", "Routine bloodwork normal"]
    }
}

def generate_patient_history(risk_level):
    """Procedurally generates a 3-visit fragmented timeline."""
    age = random.randint(18, 38)
    pool = symptom_pools[risk_level]

    # Pick 3 random past visits from different specialties to simulate "fragmentation"
    visits = []

    # Visit 1: 2 years ago
    v1_spec = "Dermatology"
    v1_symp = random.choice(pool[v1_spec])
    visits.append(f"[2022 - {v1_spec}]: Patient presented with {v1_symp}.")

    # Visit 2: 1 year ago
    # ---> THE FIX IS ON THE LINE BELOW (Changed "Family" to "Endocrinology/Family") <---
    v2_spec = "OBGYN" if risk_level != "LOW" else random.choice(["OBGYN", "Endocrinology/Family"])
    v2_symp = random.choice(pool[v2_spec])
    visits.append(f"[2023 - {v2_spec}]: Patient reported {v2_symp}.")

    # Visit 3: 3 months ago
    v3_spec = "Endocrinology/Family"
    v3_symp = random.choice(pool[v3_spec])
    visits.append(f"[Early 2024 - {v3_spec}]: Patient complaint: {v3_symp}.")

    history_str = "\n".join(visits)
    return age, history_str

def generate_portal_message(history_str, risk_level):
    """Uses the LLM to write a messy, natural portal message based on the history."""

    prompt = f"""<|im_start|>system
You are roleplaying as a female patient writing a message in a medical portal.
Read your medical history below. Write a short, natural, slightly frustrated message (2-3 sentences) asking the doctor for help.
Do NOT use medical jargon. Do NOT mention the years or doctor names. Just complain about how the symptoms are affecting you right now.<|im_end|>
<|im_start|>user
My Medical History:
{history_str}
<|im_end|>
<|im_start|>assistant
"""
    response = generator(prompt, max_new_tokens=60, temperature=0.7, return_full_text=False)
    return response[0]['generated_text'].strip().replace('"', '')



--- 2. Defining the Clinical PMOS Logic (From the Lancet Guidelines) ---


In [ ]:
print("--- 3. Generating the Synthetic Dataset ---")
# Set this to 2000 for the final run. (Keep it at 10 for your first test!)
NUM_PATIENTS = 10

# Strict Distribution: 40% Low, 45% Medium, 15% High
distribution = (
    ["LOW"] * int(NUM_PATIENTS * 0.40) +
    ["MEDIUM"] * int(NUM_PATIENTS * 0.45) +
    ["HIGH"] * int(NUM_PATIENTS * 0.15)
)
random.shuffle(distribution)

dataset = []

for idx, risk_level in enumerate(tqdm(distribution, desc="Generating Patients")):
    # 1. Build the procedural hidden history
    age, history = generate_patient_history(risk_level)

    # 2. Generate the LLM portal message based on that history
    portal_message = generate_portal_message(history, risk_level)

    dataset.append({
        "patient_id": f"PT_{1000 + idx}",
        "age": age,
        "past_medical_history": history,
        "current_portal_message": portal_message,
        "gt_risk_level": risk_level
    })



--- 3. Generating the Synthetic Dataset ---


Generating Patients:   0%|          | 0/9 [00:00<?, ?it/s][transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer Qwen2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force 

In [ ]:
print("\n--- 4. Saving to CSV ---")
df = pd.DataFrame(dataset)
df.to_csv("pmos_longitudinal_dataset.csv", index=False)
print(f"Successfully generated {len(df)} longitudinal patient records!")

# Let's peek at one of the HIGH risk patients:
high_risk_example = df[df['gt_risk_level'] == 'HIGH'].iloc[0]
print("\n--- EXAMPLE HIGH RISK PATIENT ---")
print(f"History:\n{high_risk_example['past_medical_history']}\n")
print(f"Portal Message:\n{high_risk_example['current_portal_message']}")

medium_risk_example = df[df['gt_risk_level'] == 'MEDIUM'].iloc[0]
print("\n--- EXAMPLE MEDIUM RISK PATIENT ---")
print(f"History:\n{medium_risk_example['past_medical_history']}\n")
print(f"Portal Message:\n{medium_risk_example['current_portal_message']}")

low_risk_example = df[df['gt_risk_level'] == 'LOW'].iloc[0]
print("\n--- EXAMPLE LOW RISK PATIENT ---")
print(f"History:\n{low_risk_example['past_medical_history']}\n")
print(f"Portal Message:\n{low_risk_example['current_portal_message']}")


--- 4. Saving to CSV ---
Successfully generated 9 longitudinal patient records!

--- EXAMPLE HIGH RISK PATIENT ---
History:
[2022 - Dermatology]: Patient presented with Severe cystic acne on jawline.
[2023 - OBGYN]: Patient reported Irregular heavy bleeding.
[Early 2024 - Endocrinology/Family]: Patient complaint: Rapid unexplained weight gain (10kg+).

Portal Message:
Hey doc, my face is so red and itchy from that acne, I can't even wear makeup anymore! And all this time I've been feeling really bloated, like I'm going to burst...I don't know what's happening to me. Can you help?

--- EXAMPLE MEDIUM RISK PATIENT ---
History:
[2022 - Dermatology]: Patient presented with Mild breakouts on chin.
[2023 - OBGYN]: Patient reported Painful cramps.
[Early 2024 - Endocrinology/Family]: Patient complaint: Difficulty losing weight.

Portal Message:
Hey Doc,

I'm feeling really down because my skin is acting up again. I've been trying to lose some weight, but it's like my body has a mind of its o

# **FULL DATASET GENERATION**

In [ ]:
import pandas as pd
import random
import torch
from transformers import pipeline
from tqdm import tqdm

print("--- 1. Initializing Local LLM for Patient Persona ---")
generator = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
    torch_dtype=torch.bfloat16
)



--- 1. Initializing Local LLM for Patient Persona ---


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
print("--- 2. Defining the Clinical PMOS Logic ---")
symptom_pools = {
    "HIGH": {
        "Dermatology": ["Severe cystic acne on jawline", "Noticeable hair thinning", "Acanthosis nigricans (dark skin patches)"],
        "OBGYN": ["Missed periods for 4-6 months", "Irregular heavy bleeding", "Ovarian cysts observed on ultrasound"],
        "Endocrinology/Family": ["Rapid unexplained weight gain (10kg+)", "Severe daily fatigue", "Borderline high HbA1c (insulin resistance)"]
    },
    "MEDIUM": {
        "Dermatology": ["Mild breakouts on chin", "Oily skin"],
        "OBGYN": ["Periods occasionally late (35-40 day cycles)", "Painful cramps"],
        "Endocrinology/Family": ["Feeling tired after meals", "Difficulty losing weight", "Sugar cravings"]
    },
    "LOW": {
        "Dermatology": ["Occasional dry skin", "Sunburn", "Normal skin check"],
        "OBGYN": ["Regular 28-day cycles", "Routine pap smear normal", "Refill birth control"],
        "Endocrinology/Family": ["Sprained ankle", "Seasonal allergies", "Mild cold symptoms", "Routine bloodwork normal"]
    }
}

import random

def generate_patient_history(risk_level):
    age = random.randint(18, 38)
    pool = symptom_pools[risk_level]
    visits = []

    # 1. Randomize the number of prior visits (from 1 to 4)
    num_visits = random.randint(1, 4)

    # 2. Add a realistic chance (10%) that the patient has NO medical history on file
    if random.random() < 0.10:
        return age, "No prior encounters in system."

    # A pool of messy, realistic EHR timestamps
    timeframes = ["2019", "2020", "2021", "2022", "2023", "4 years ago", "18 months ago", "Last summer", "6 months ago"]

    # Pick random times and sort them so the timeline flows chronologically
    patient_times = sorted(random.sample(timeframes, num_visits), key=lambda x: str(x))

    for time_label in patient_times:
        # 3. Completely randomize which doctor they saw
        # They might see the OBGYN twice and never see a Dermatologist!
        available_specialties = list(pool.keys())
        chosen_spec = random.choice(available_specialties)

        # Pick the symptom
        chosen_symp = random.choice(pool[chosen_spec])

        # 4. Add variability to how the doctor typed the note
        phrasing = random.choice([
            f"[{time_label} - {chosen_spec}]: Patient presented with {chosen_symp}.",
            f"[{time_label} | {chosen_spec}] Note: {chosen_symp}.",
            f"Date: {time_label} | Dept: {chosen_spec} | CC: {chosen_symp}"
        ])

        visits.append(phrasing)

    return age, "\n".join(visits)

def generate_portal_message(history_str, risk_level):
    prompt = f"""<|im_start|>system
You are a distracted, stressed patient writing a message in a medical portal.
Read your medical history below, but DO NOT mention all of it.
Rules for your message:
1. Only complain about ONE or TWO of the symptoms. Completely ignore the rest.
2. Be vague. Use non-medical words (e.g., "breaking out" instead of "acne", "putting on pounds" instead of "weight gain").
3. Add 1 sentence of completely irrelevant "clinical noise" (e.g., stress at work, a sick kid, bad weather, or a random headache).
Keep it under 3 sentences.<|im_end|>
<|im_start|>user
My Medical History:
{history_str}
<|im_end|>
<|im_start|>assistant
"""
    response = generator(prompt, max_new_tokens=70, temperature=0.8, return_full_text=False)
    return response[0]['generated_text'].strip().replace('"', '')



--- 2. Defining the Clinical PMOS Logic ---


In [10]:
print("--- 3. Generating the MASTER Synthetic Dataset (3,000 Records) ---")

NUM_PATIENTS = 3000

# Perfect Mathematical Distribution
distribution = (
    ["LOW"] * int(NUM_PATIENTS * 0.40) +    # 800 Patients
    ["MEDIUM"] * int(NUM_PATIENTS * 0.45) + # 900 Patients
    ["HIGH"] * int(NUM_PATIENTS * 0.15)     # 300 Patients
)
random.shuffle(distribution)

dataset = []

for idx, risk_level in enumerate(tqdm(distribution, desc="Generating Patients")):
    age, history = generate_patient_history(risk_level)
    portal_message = generate_portal_message(history, risk_level)

    dataset.append({
        "patient_id": f"PT_{1000 + idx}",
        "age": age,
        "past_medical_history": history,
        "current_portal_message": portal_message,
        "gt_risk_level": risk_level
    })

    # --- AUTO-SAVE FEATURE (Saves every 100 patients) ---
    if (idx + 1) % 100 == 0:
        pd.DataFrame(dataset).to_csv("pmos_dataset_BACKUP.csv", index=False)



--- 3. Generating the MASTER Synthetic Dataset (3,000 Records) ---


Generating Patients:   0%|          | 10/3000 [00:16<1:20:57,  1.62s/it][transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
[transformers] Both `max_new_tokens` (=70) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Generating Patients:  99%|█████████▉| 2980/3000 [1:34:22<00:36,  1.82s/it][transformers] Both `max_new_tokens` (=70) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=70) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.

In [11]:
print("\n--- 4. Saving Final CSV ---")
df = pd.DataFrame(dataset)

df.to_csv("pmos_longitudinal_dataset_FINAL.csv", index=False)
print(f"SUCCESS! {len(df)} longitudinal patient records saved to 'pmos_longitudinal_dataset_FINAL.csv'.")

# Rename the column from 'Risk' to 'Clinical Priority' to reflect a retrospective Triage process
if 'gt_risk_level' in df.columns:
    df.rename(columns={'gt_risk_level': 'clinical_priority'}, inplace=True)

# Print the new values to verify the change
print("Updated columns:", df.columns.tolist())
print(df['clinical_priority'].value_counts())
df.to_csv("pmos_longitudinal_dataset_FINALformodel.csv", index=False)
print(f"SUCCESS! {len(df)} longitudinal patient records saved to 'pmos_longitudinal_dataset_FINAL.csv'.")



--- 4. Saving Final CSV ---
SUCCESS! 3000 longitudinal patient records saved to 'pmos_longitudinal_dataset_FINAL.csv'.
Updated columns: ['patient_id', 'age', 'past_medical_history', 'current_portal_message', 'clinical_priority']
clinical_priority
MEDIUM    1350
LOW       1200
HIGH       450
Name: count, dtype: int64
SUCCESS! 3000 longitudinal patient records saved to 'pmos_longitudinal_dataset_FINAL.csv'.

--- 4. Saving Final CSV ---
SUCCESS! 3000 longitudinal patient records saved to 'pmos_longitudinal_dataset_FINAL.csv'.
Updated columns: ['patient_id', 'age', 'past_medical_history', 'current_portal_message', 'clinical_priority']
clinical_priority
MEDIUM    1350
LOW       1200
HIGH       450
Name: count, dtype: int64
SUCCESS! 3000 longitudinal patient records saved to 'pmos_longitudinal_dataset_FINAL.csv'.

--- 4. Saving Final CSV ---
SUCCESS! 3000 longitudinal patient records saved to 'pmos_longitudinal_dataset_FINAL.csv'.
Updated columns: ['patient_id', 'age', 'past_medical_history